In [1]:
#1
!pip install -q tensorflow

#  Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#2
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import random
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import Conv1D, GlobalAveragePooling1D
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score, precision_score, recall_score)
from sklearn.model_selection import train_test_split


np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print(f" TensorFlow version: {tf.__version__}")
print(f" GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

 TensorFlow version: 2.20.0
 GPU available: True


In [3]:
!ls /content/drive/MyDrive/GP/CSV_completeDataset

testing  training  valid


In [4]:
def load_labeled_data(base_path, window_size=10, stride=2):
    X_windows = []
    y_windows = []
    video_ids_windows = [] # New list to store video IDs for each window

    categories = {
        'Drowsy': 1,
        'Alert': 0
    }

    for category, label_value in categories.items():
        folder_path = os.path.join(base_path, category)

        if not os.path.exists(folder_path):
            print(f"Warning: Folder not found: {folder_path}")
            continue

        for filename in os.listdir(folder_path):
            if filename.endswith(".csv"):
                try:
                    file_path = os.path.join(folder_path, filename)
                    df = pd.read_csv(file_path)

                    features = df[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values
                    row_labels = np.full((features.shape[0],), label_value)

                    if len(features) < window_size:
                        continue

                    # Use the filename as a unique video ID for the current file
                    video_id = filename

                    for i in range(0, len(features) - window_size + 1, stride):
                        window_features = features[i:i + window_size]
                        window_labels = row_labels[i:i + window_size]

                        mode_result = stats.mode(window_labels, keepdims=True)
                        final_label = mode_result.mode[0]

                        X_windows.append(window_features)
                        y_windows.append(final_label)
                        video_ids_windows.append(video_id) # Append video ID for each window

                except Exception as e:
                    print(f"Error processing {filename}: {e}")

    return np.array(X_windows), np.array(y_windows), np.array(video_ids_windows)

In [5]:
!ls '/content/drive/MyDrive/GP/CSV_completeDataset/'

testing  training  valid


In [6]:
train_base = '/content/drive/MyDrive/GP/CSV_completeDataset/training'
test_base   = '/content/drive/MyDrive/GP/CSV_completeDataset/testing'
val_base  = '/content/drive/MyDrive/GP/CSV_completeDataset/valid'

WINDOW_SIZE = 20
STRIDE = 2

X_train_raw, y_train, video_ids_train_raw = load_labeled_data(train_base, WINDOW_SIZE, STRIDE)
X_val_raw, y_val, video_ids_val = load_labeled_data(val_base, WINDOW_SIZE, STRIDE)
X_test_raw, y_test, video_ids_test_raw = load_labeled_data(test_base, WINDOW_SIZE, STRIDE)

print("Train:", X_train_raw.shape, y_train.shape, video_ids_train_raw.shape)
print("Val:  ", X_val_raw.shape, y_val.shape, video_ids_val.shape)
print("Test: ", X_test_raw.shape, y_test.shape, video_ids_test_raw.shape)

if X_train_raw.ndim < 3:
    print("❌ ERROR: No training windows were created!")
else:
    print(f"✅ Total Training Windows: {X_train_raw.shape[0]}")
    print(f"✅ Window Shape: {X_train_raw.shape[1]} rows x {X_train_raw.shape[2]} features")

Train: (1554, 20, 4) (1554,) (1554,)
Val:   (277, 20, 4) (277,) (277,)
Test:  (577, 20, 4) (577,) (577,)
✅ Total Training Windows: 1554
✅ Window Shape: 20 rows x 4 features


In [7]:
scaler = StandardScaler()

# TRAIN
X_train_reshaped = X_train_raw.reshape(-1, 4)
X_train_scaled = scaler.fit_transform(X_train_reshaped)
X_train = X_train_scaled.reshape(-1, WINDOW_SIZE, 4)

# VALIDATION
X_val_reshaped = X_val_raw.reshape(-1, 4)
X_val_scaled = scaler.transform(X_val_reshaped)
X_val = X_val_scaled.reshape(-1, WINDOW_SIZE, 4)

# TEST
X_test_reshaped = X_test_raw.reshape(-1, 4)
X_test_scaled = scaler.transform(X_test_reshaped)
X_test = X_test_scaled.reshape(-1, WINDOW_SIZE, 4)

print("Scaled Train:", X_train.shape)
print("Scaled Val:  ", X_val.shape)
print("Scaled Test: ", X_test.shape)

Scaled Train: (1554, 20, 4)
Scaled Val:   (277, 20, 4)
Scaled Test:  (577, 20, 4)


In [ ]:
gru_model = Sequential([
    GRU(128, return_sequences=True, input_shape=(WINDOW_SIZE, 4)),
    GRU(16),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

gru_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_44"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_44 (GRU)                    │ (None, 20, 128)        │        51,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_45 (GRU)                    │ (None, 16)             │         7,008 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_88 (Dropout)            │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_88 (Dense)                │ (None, 16)             │           272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_89 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 58,753 (229.50 KB)

 Trainable params: 58,753 (229.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
gru_history = gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    verbose=1
)

gru_save_path = '/content/drive/MyDrive/GP/LstmModels/best_gru_drowsy_model.h5'
gru_model.save(gru_save_path)

print(f"🚀 GRU model saved successfully to: {gru_save_path}")

Epoch 1/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 8s 47ms/step - accuracy: 0.5927 - loss: 0.6769 - val_accuracy: 0.6643 - val_loss: 0.6683
Epoch 2/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.8179 - loss: 0.6105 - val_accuracy: 0.7617 - val_loss: 0.6082
Epoch 3/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8752 - loss: 0.5005 - val_accuracy: 0.7978 - val_loss: 0.5032
Epoch 4/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9254 - loss: 0.3219 - val_accuracy: 0.8087 - val_loss: 0.4449
Epoch 5/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.9434 - loss: 0.1910 - val_accuracy: 0.8123 - val_loss: 0.4838
Epoch 6/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9575 - loss: 0.1505 - val_accuracy: 0.8159 - val_loss: 0.5125
Epoch 7/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9614 - loss: 0.1269 - val_accuracy: 0.8195 - val_loss: 0.5392
Epoch 8/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9672 - loss: 0.1117 - val_accuracy: 0.8231 - v

🚀 GRU model saved successfully to: /content/drive/MyDrive/GP/LstmModels/best_gru_drowsy_model.h5


In [ ]:
tcn_model = Sequential([
    Conv1D(64, kernel_size=3, dilation_rate=1, padding='causal', activation='relu', input_shape=(WINDOW_SIZE, 4)),
    Dropout(0.3),

    Conv1D(64, kernel_size=3, dilation_rate=2, padding='causal', activation='relu'),
    Dropout(0.3),

    Conv1D(32, kernel_size=3, dilation_rate=4, padding='causal', activation='relu'),
    Dropout(0.3),

    GlobalAveragePooling1D(),

    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

tcn_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

tcn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_45"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_66 (Conv1D)              │ (None, 20, 64)         │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_89 (Dropout)            │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_67 (Conv1D)              │ (None, 20, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_90 (Dropout)            │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_68 (Conv1D)              │ (None, 20, 32)         │         6,176 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_91 (Dropout)            │ (None, 20, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_22     │ (None, 32)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_90 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_91 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,905 (77.75 KB)

 Trainable params: 19,905 (77.75 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
tcn_history = tcn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=20,
    batch_size=32,
    verbose=1
)

tcn_save_path = '/content/drive/MyDrive/GP/LstmModels/best_tcn_drowsy_model.h5'
tcn_model.save(tcn_save_path)

print(f"🚀 TCN model saved successfully to: {tcn_save_path}")

Epoch 1/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 10s 83ms/step - accuracy: 0.5727 - loss: 0.6842 - val_accuracy: 0.5596 - val_loss: 0.6752
Epoch 2/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6287 - loss: 0.6485 - val_accuracy: 0.6643 - val_loss: 0.6285
Epoch 3/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8308 - loss: 0.5705 - val_accuracy: 0.8014 - val_loss: 0.5422
Epoch 4/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9073 - loss: 0.4489 - val_accuracy: 0.8195 - val_loss: 0.4518
Epoch 5/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9221 - loss: 0.3294 - val_accuracy: 0.8267 - val_loss: 0.4065
Epoch 6/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9356 - loss: 0.2478 - val_accuracy: 0.8231 - val_loss: 0.4055
Epoch 7/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9414 - loss: 0.2042 - val_accuracy: 0.8195 - val_loss: 0.4298
Epoch 8/20
49/49 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9479 - loss: 0.1753 - val_accuracy: 0.8195 - val_los

🚀 TCN model saved successfully to: /content/drive/MyDrive/GP/LstmModels/best_tcn_drowsy_model.h5


In [ ]:
def evaluate_model_by_file(model, model_name, test_base, scaler, window_size=10, stride=2):
    total_windows_tested = 0
    correct_windows = 0

    total_videos_tested = 0
    correct_videos = 0

    print(f"\n========== {model_name} RESULTS ==========")
    print(f"{'File Name':<25} | {'Label':<8} | {'Window Acc':<12} | {'Drowsy %':<10} | {'Verdict':<8} | {'Status'}")
    print("-" * 95)

    for category in ['Drowsy', 'Alert']:
        folder_path = os.path.join(test_base, category)

        if not os.path.exists(folder_path):
            print(f"Warning: Folder not found: {folder_path}")
            continue

        true_val = 1 if category == 'Drowsy' else 0

        for filename in os.listdir(folder_path):
            if filename.endswith(".csv"):
                file_path = os.path.join(folder_path, filename)
                df_file = pd.read_csv(file_path)

                feats = df_file[['Duration', 'Amplitude', 'Velocity', 'Frequency']].values

                if len(feats) < window_size:
                    continue

                file_windows = []

                for i in range(0, len(feats) - window_size + 1, stride):
                    file_windows.append(feats[i:i + window_size])

                X_file = np.array(file_windows)

                X_file_reshaped = X_file.reshape(-1, 4)
                X_file_scaled = scaler.transform(X_file_reshaped)
                X_file_final = X_file_scaled.reshape(-1, window_size, 4)

                file_preds = model.predict(X_file_final, verbose=0)
                file_rounded = (file_preds > 0.5).astype(int).flatten()

                # Window accuracy for this file
                file_window_acc = np.mean(file_rounded == true_val) * 100

                total_windows_tested += len(file_rounded)
                correct_windows += np.sum(file_rounded == true_val)

                # Video-level decision
                drowsy_percent = (np.sum(file_rounded) / len(file_rounded)) * 100
                verdict_val = 1 if drowsy_percent > 50 else 0

                total_videos_tested += 1

                if verdict_val == true_val:
                    correct_videos += 1

                status = "✅" if verdict_val == true_val else "❌"
                verdict_text = "DROWSY" if verdict_val == 1 else "ALERT"

                print(
                    f"{filename[:25]:<25} | "
                    f"{category:<8} | "
                    f"{file_window_acc:>9.1f}% | "
                    f"{drowsy_percent:>8.1f}% | "
                    f"{verdict_text:<8} | "
                    f"{status}"
                )

    window_accuracy = (correct_windows / total_windows_tested) * 100 if total_windows_tested > 0 else 0
    video_accuracy = (correct_videos / total_videos_tested) * 100 if total_videos_tested > 0 else 0

    print("\n" + "=" * 45)
    print(f"{model_name} TOTAL WINDOW ACCURACY: {window_accuracy:.2f}%")
    print(f"{model_name} TOTAL VIDEO ACCURACY:  {video_accuracy:.2f}%")
    print("=" * 45)

    return window_accuracy, video_accuracy

In [ ]:
gru_window_acc, gru_video_acc = evaluate_model_by_file(
    gru_model,
    "GRU",
    test_base,
    scaler,
    WINDOW_SIZE,
    STRIDE
)

tcn_window_acc, tcn_video_acc = evaluate_model_by_file(
    tcn_model,
    "TCN",
    test_base,
    scaler,
    WINDOW_SIZE,
    STRIDE
)


========== GRU RESULTS ==========
File Name                 | Label    | Window Acc   | Drowsy %   | Verdict  | Status
-----------------------------------------------------------------------------------------------
D022_20260513_190626_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D023_20260513_195650_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D024_20260513_212814_fram | Drowsy   |      93.5% |     93.5% | DROWSY   | ✅
D025_20260513_233342_fram | Drowsy   |      25.0% |     25.0% | ALERT    | ❌
D026_20260514_025307_fram | Drowsy   |      87.3% |     87.3% | DROWSY   | ✅
A022_20260513_190541_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A023_20260513_195602_fram | Alert    |      76.8% |     23.2% | ALERT    | ✅
A024_20260513_212736_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A025_20260513_233235_fram | Alert    |      87.6% |     12.4% | ALERT    | ✅
A026_20260514_025227_fram | Alert    |      98.7% |      1.3% | ALERT    | ✅

GRU TOTAL WIN

In [ ]:
!pip install optuna

In [ ]:
import optuna
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GRU,
    Dense,
    Dropout
)

In [ ]:
def objective_gru(trial):

    # Hyperparameters to tune
    gru_units_1 = trial.suggest_int("gru_units_1", 32, 128)
    gru_units_2 = trial.suggest_int("gru_units_2", 8, 64)

    dropout_rate = trial.suggest_float("dropout", 0.1, 0.5)

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-5,
        1e-3,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [16, 32, 64]
    )

    # Build model
    model = Sequential([

        GRU(
            gru_units_1,
            return_sequences=True,
            input_shape=(WINDOW_SIZE, 4)
        ),

        GRU(gru_units_2),

        Dropout(dropout_rate),

        Dense(16, activation='relu'),

        Dense(1, activation='sigmoid')

    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # Train
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=batch_size,
        verbose=0
    )

    # Return BEST validation accuracy
    best_val_acc = max(history.history['val_accuracy'])

    return best_val_acc

In [ ]:
study_gru = optuna.create_study(direction="maximize")

study_gru.optimize(
    objective_gru,
    n_trials=20
)

[I 2026-06-02 12:05:02,393] A new study created in memory with name: no-name-bbd541b0-c111-4033-8d77-30a2c2d1207f
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
[I 2026-06-02 12:05:29,580] Trial 0 finished with value: 0.8303248882293701 and parameters: {'gru_units_1': 87, 'gru_units_2': 21, 'dropout': 0.1801192285823516, 'learning_rate': 6.340961327019132e-05, 'batch_size': 16}. Best is trial 0 with value: 0.8303248882293701.
[I 2026-06-02 12:05:40,095] Trial 1 finished with value: 0.8231046795845032 and parameters: {'gru_units_1': 97, 'gru_units_2': 47, 'dropout': 0.23155507209346293, 'learning_rate': 0.000622281437501698, 'batch_size': 32}. Best is trial 0 with value: 0.8303248882293701.
[I 2026-06-02 12:05:49,544] Trial 2 finished with value: 0.841

In [ ]:
print("Best Trial:")
print(study_gru.best_trial)

print("\nBest Params:")
print(study_gru.best_params)

print("\nBest Validation Accuracy:")
print(study_gru.best_value)

Best Trial:
FrozenTrial(number=2, state=<TrialState.COMPLETE: 1>, values=[0.8411552309989929], datetime_start=datetime.datetime(2026, 6, 2, 12, 5, 40, 97592), datetime_complete=datetime.datetime(2026, 6, 2, 12, 5, 49, 544874), params={'gru_units_1': 41, 'gru_units_2': 49, 'dropout': 0.17577537013775327, 'learning_rate': 0.0002623867394540794, 'batch_size': 32}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'gru_units_1': IntDistribution(high=128, log=False, low=32, step=1), 'gru_units_2': IntDistribution(high=64, log=False, low=8, step=1), 'dropout': FloatDistribution(high=0.5, log=False, low=0.1, step=None), 'learning_rate': FloatDistribution(high=0.001, log=True, low=1e-05, step=None), 'batch_size': CategoricalDistribution(choices=(16, 32, 64))}, trial_id=2, value=None)

Best Params:
{'gru_units_1': 41, 'gru_units_2': 49, 'dropout': 0.17577537013775327, 'learning_rate': 0.0002623867394540794, 'batch_size': 32}

Best Validation Accuracy:
0.8411552309989929


In [ ]:
best_gru_model = Sequential([
    GRU(88, return_sequences=True, input_shape=(WINDOW_SIZE, 4)),
    GRU(30),
    Dropout(0.4176403928249539),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

best_gru_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=5.8407380914947864e-05
    ),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

best_gru_history = best_gru_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=16,
    verbose=1
)

save_path = '/content/drive/MyDrive/GP/LstmModels/final_best_gru_optuna.h5'
best_gru_model.save(save_path)

print(f"🚀 Final tuned GRU model saved to: {save_path}")

Epoch 1/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.5695 - loss: 0.6861 - val_accuracy: 0.6787 - val_loss: 0.6788
Epoch 2/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.7902 - loss: 0.6417 - val_accuracy: 0.7726 - val_loss: 0.6380
Epoch 3/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8694 - loss: 0.5735 - val_accuracy: 0.8123 - val_loss: 0.5763
Epoch 4/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9060 - loss: 0.4564 - val_accuracy: 0.8267 - val_loss: 0.4547
Epoch 5/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.9311 - loss: 0.2729 - val_accuracy: 0.8123 - val_loss: 0.4028
Epoch 6/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9511 - loss: 0.1729 - val_accuracy: 0.8267 - val_loss: 0.4464
Epoch 7/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.9588 - loss: 0.1440 - val_accuracy: 0.8195 - val_loss: 0.4702
Epoch 8/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.9627 - loss: 0.1242 - val_accuracy: 0.8195 - v

🚀 Final tuned GRU model saved to: /content/drive/MyDrive/GP/LstmModels/final_best_gru_optuna.h5


In [ ]:
gru_window_acc, gru_video_acc = evaluate_model_by_file(
    best_gru_model,
    "FINAL OPTUNA GRU",
    test_base,
    scaler,
    WINDOW_SIZE,
    STRIDE
)


========== FINAL OPTUNA GRU RESULTS ==========
File Name                 | Label    | Window Acc   | Drowsy %   | Verdict  | Status
-----------------------------------------------------------------------------------------------
D022_20260513_190626_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D023_20260513_195650_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D024_20260513_212814_fram | Drowsy   |      96.1% |     96.1% | DROWSY   | ✅
D025_20260513_233342_fram | Drowsy   |      25.0% |     25.0% | ALERT    | ❌
D026_20260514_025307_fram | Drowsy   |      87.3% |     87.3% | DROWSY   | ✅
A022_20260513_190541_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A023_20260513_195602_fram | Alert    |      76.8% |     23.2% | ALERT    | ✅
A024_20260513_212736_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A025_20260513_233235_fram | Alert    |      84.3% |     15.7% | ALERT    | ✅
A026_20260514_025227_fram | Alert    |      97.3% |      2.7% | ALERT    | ✅



In [ ]:
def objective_tcn(trial):

    filters_1 = trial.suggest_categorical("filters_1", [32, 64, 96, 128])
    filters_2 = trial.suggest_categorical("filters_2", [32, 64, 96])
    filters_3 = trial.suggest_categorical("filters_3", [16, 32, 64])

    kernel_size = trial.suggest_categorical("kernel_size", [2, 3, 5])

    dropout_rate = trial.suggest_float("dropout", 0.1, 0.5)

    dense_units = trial.suggest_categorical("dense_units", [8, 16, 32])

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-5,
        1e-3,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [16, 32, 64]
    )

    model = Sequential([
        Conv1D(
            filters_1,
            kernel_size=kernel_size,
            dilation_rate=1,
            padding='causal',
            activation='relu',
            input_shape=(WINDOW_SIZE, 4)
        ),
        Dropout(dropout_rate),

        Conv1D(
            filters_2,
            kernel_size=kernel_size,
            dilation_rate=2,
            padding='causal',
            activation='relu'
        ),
        Dropout(dropout_rate),

        Conv1D(
            filters_3,
            kernel_size=kernel_size,
            dilation_rate=4,
            padding='causal',
            activation='relu'
        ),
        Dropout(dropout_rate),

        GlobalAveragePooling1D(),

        Dense(dense_units, activation='relu'),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=10,
        batch_size=batch_size,
        verbose=0
    )

    best_val_acc = max(history.history['val_accuracy'])

    return best_val_acc

In [ ]:
study_tcn = optuna.create_study(direction="maximize")

study_tcn.optimize(
    objective_tcn,
    n_trials=20
)

[I 2026-06-02 12:09:45,790] A new study created in memory with name: no-name-e0c18925-362c-4c82-8e6d-5eca40ea561f
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
[I 2026-06-02 12:09:58,728] Trial 0 finished with value: 0.6642599105834961 and parameters: {'filters_1': 128, 'filters_2': 32, 'filters_3': 64, 'kernel_size': 2, 'dropout': 0.27855093172530165, 'dense_units': 8, 'learning_rate': 2.8694029555785238e-05, 'batch_size': 32}. Best is trial 0 with value: 0.6642599105834961.
[I 2026-06-02 12:10:12,057] Trial 1 finished with value: 0.8014440536499023 and parameters: {'filters_1': 128, 'filters_2': 96, 'filters_3': 16, 'kernel_size': 5, 'dropout': 0.2345185239161074, 'dense_units': 32, 'learni

In [ ]:
print("Best TCN Trial:")
print(study_tcn.best_trial)

print("\nBest TCN Params:")
print(study_tcn.best_params)

print("\nBest TCN Validation Accuracy:")
print(study_tcn.best_value)

Best TCN Trial:
FrozenTrial(number=12, state=<TrialState.COMPLETE: 1>, values=[0.6949740052223206], datetime_start=datetime.datetime(2026, 6, 2, 11, 59, 18, 957170), datetime_complete=datetime.datetime(2026, 6, 2, 11, 59, 33, 64493), params={'filters_1': 32, 'filters_2': 32, 'filters_3': 16, 'kernel_size': 5, 'dropout': 0.49876496979911766, 'dense_units': 16, 'learning_rate': 0.0008934228053411201, 'batch_size': 16}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'filters_1': CategoricalDistribution(choices=(32, 64, 96, 128)), 'filters_2': CategoricalDistribution(choices=(32, 64, 96)), 'filters_3': CategoricalDistribution(choices=(16, 32, 64)), 'kernel_size': CategoricalDistribution(choices=(2, 3, 5)), 'dropout': FloatDistribution(high=0.5, log=False, low=0.1, step=None), 'dense_units': CategoricalDistribution(choices=(8, 16, 32)), 'learning_rate': FloatDistribution(high=0.001, log=True, low=1e-05, step=None), 'batch_size': CategoricalDistribution(choices=(16, 3

In [ ]:
best_tcn_model = Sequential([
    Conv1D(
        32,
        kernel_size=5,
        dilation_rate=1,
        padding='causal',
        activation='relu',
        input_shape=(WINDOW_SIZE, 4)
    ),
    Dropout(0.2986960433074547),

    Conv1D(
        96,
        kernel_size=5,
        dilation_rate=2,
        padding='causal',
        activation='relu'
    ),
    Dropout(0.2986960433074547),

    Conv1D(
        32,
        kernel_size=5,
        dilation_rate=4,
        padding='causal',
        activation='relu'
    ),
    Dropout(0.2986960433074547),

    GlobalAveragePooling1D(),

    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

best_tcn_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.0009854876557513254
    ),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

best_tcn_history = best_tcn_model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=16,
    verbose=1
)

save_path = '/content/drive/MyDrive/GP/LstmModels/final_best_tcn_optuna.h5'
best_tcn_model.save(save_path)

print(f"🚀 Final tuned TCN model saved to: {save_path}")

Epoch 1/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 9s 45ms/step - accuracy: 0.8533 - loss: 0.3351 - val_accuracy: 0.8556 - val_loss: 0.5097
Epoch 2/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9505 - loss: 0.1334 - val_accuracy: 0.8412 - val_loss: 0.6554
Epoch 3/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9614 - loss: 0.0962 - val_accuracy: 0.8303 - val_loss: 0.7880
Epoch 4/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9710 - loss: 0.0740 - val_accuracy: 0.8303 - val_loss: 0.9639
Epoch 5/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9685 - loss: 0.0784 - val_accuracy: 0.8303 - val_loss: 0.9429
Epoch 6/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9691 - loss: 0.0780 - val_accuracy: 0.8303 - val_loss: 1.1339
Epoch 7/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9755 - loss: 0.0616 - val_accuracy: 0.8267 - val_loss: 1.1516
Epoch 8/30
98/98 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9801 - loss: 0.0487 - val_accuracy: 0.8231 - val_loss

🚀 Final tuned TCN model saved to: /content/drive/MyDrive/GP/LstmModels/final_best_tcn_optuna.h5


In [ ]:
tcn_window_acc, tcn_video_acc = evaluate_model_by_file(
    best_tcn_model,
    "FINAL OPTUNA TCN",
    test_base,
    scaler,
    WINDOW_SIZE,
    STRIDE
)


========== FINAL OPTUNA TCN RESULTS ==========
File Name                 | Label    | Window Acc   | Drowsy %   | Verdict  | Status
-----------------------------------------------------------------------------------------------
D022_20260513_190626_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D023_20260513_195650_fram | Drowsy   |     100.0% |    100.0% | DROWSY   | ✅
D024_20260513_212814_fram | Drowsy   |      87.0% |     87.0% | DROWSY   | ✅
D025_20260513_233342_fram | Drowsy   |      15.4% |     15.4% | ALERT    | ❌
D026_20260514_025307_fram | Drowsy   |      89.1% |     89.1% | DROWSY   | ✅
A022_20260513_190541_fram | Alert    |       7.5% |     92.5% | DROWSY   | ❌
A023_20260513_195602_fram | Alert    |      82.6% |     17.4% | ALERT    | ✅
A024_20260513_212736_fram | Alert    |       0.0% |    100.0% | DROWSY   | ❌
A025_20260513_233235_fram | Alert    |     100.0% |      0.0% | ALERT    | ✅
A026_20260514_025227_fram | Alert    |      94.7% |      5.3% | ALERT    | ✅



In [ ]:
print("GRU Window Accuracy:", gru_window_acc)
print("GRU Video Accuracy:", gru_video_acc)

print("TCN Window Accuracy:", tcn_window_acc)
print("TCN Video Accuracy:", tcn_video_acc)

GRU Window Accuracy: 68.1109185441941
GRU Video Accuracy: 70.0
TCN Window Accuracy: 69.67071057192375
TCN Video Accuracy: 70.0


In [ ]:
import pandas as pd

def optuna_top10_table(study, model_name="Model"):
    rows = []

    for trial in study.trials:
        if trial.value is not None:
            row = trial.params.copy()
            row["val_accuracy"] = trial.value
            rows.append(row)

    df = pd.DataFrame(rows)

    df = df.sort_values(by="val_accuracy", ascending=False).head(10)

    print(f"Top 10 Bayesian Optimization Results for {model_name}")
    display(df)

    return df

In [ ]:
gru_top10 = optuna_top10_table(study_gru, "GRU")

Top 10 Bayesian Optimization Results for GRU


,gru_units_1,gru_units_2,dropout,learning_rate,batch_size,val_accuracy
2,41,49,0.175775,0.000262,32,0.841155
6,122,64,0.282085,0.000059,16,0.837545
15,71,43,0.402768,0.000160,32,0.833935
0,87,21,0.180119,0.000063,16,0.830325
17,45,47,0.265512,0.000131,32,0.830325
3,74,53,0.163686,0.000104,16,0.830325
13,60,54,0.446997,0.000227,16,0.826715
1,97,47,0.231555,0.000622,32,0.823105
8,108,26,0.332259,0.000824,64,0.823105
4,57,18,0.359418,0.000823,64,0.823105


In [ ]:
tcn_top10 = optuna_top10_table(study_tcn, "TCN")

Top 10 Bayesian Optimization Results for TCN


,filters_1,filters_2,filters_3,kernel_size,dropout,dense_units,learning_rate,batch_size,val_accuracy
11,96,64,64,2,0.470592,16,0.000980,16,0.851986
10,96,64,64,2,0.453771,16,0.000759,16,0.844765
16,96,64,64,3,0.443677,16,0.000485,16,0.844765
4,96,96,64,2,0.460049,16,0.000066,16,0.841155
14,96,64,64,2,0.473626,16,0.000446,16,0.837545
17,96,64,64,2,0.363509,16,0.000460,16,0.837545
12,96,64,64,2,0.426337,16,0.000880,16,0.837545
5,32,96,32,3,0.321410,8,0.000260,32,0.837545
18,96,64,64,2,0.496185,16,0.000984,16,0.833935
13,96,64,64,2,0.408969,16,0.000876,16,0.830325


In [ ]:
def objective_gru(trial):

    gru_units_1 = trial.suggest_int("gru_units_1", 32, 128)
    gru_units_2 = trial.suggest_int("gru_units_2", 16, 64)
    dense_units = trial.suggest_categorical("dense_units", [8, 16, 24, 32])
    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-3, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    model = Sequential([
        GRU(gru_units_1, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
        Dropout(dropout),

        GRU(gru_units_2),
        Dropout(dropout),

        Dense(dense_units, activation="relu"),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = EarlyStopping(
        monitor="val_accuracy",
        patience=5,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=0
    )

    val_accuracy = max(history.history["val_accuracy"])
    return val_accuracy

In [ ]:
gru_study = optuna.create_study(direction="maximize")
gru_study.optimize(objective_gru, n_trials=20)

print("Best GRU Trial:")
print(gru_study.best_trial)

[I 2026-06-02 12:42:38,113] A new study created in memory with name: no-name-10ecd80b-c02c-408d-b097-346897067f48
/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
[I 2026-06-02 12:42:50,053] Trial 0 finished with value: 0.826714813709259 and parameters: {'gru_units_1': 71, 'gru_units_2': 26, 'dense_units': 24, 'dropout': 0.2587297011307377, 'learning_rate': 9.192121252582893e-05, 'batch_size': 32}. Best is trial 0 with value: 0.826714813709259.
[I 2026-06-02 12:43:13,821] Trial 1 finished with value: 0.8231046795845032 and parameters: {'gru_units_1': 92, 'gru_units_2': 34, 'dense_units': 24, 'dropout': 0.4857199099910047, 'learning_rate': 1.0117017886044039e-05, 'batch_size': 16}. Best is trial 0 with value: 0.826714813709259.
[I 2026-06-02 12:43:39,070

Best GRU Trial:
FrozenTrial(number=14, state=<TrialState.COMPLETE: 1>, values=[0.8411552309989929], datetime_start=datetime.datetime(2026, 6, 2, 12, 46, 4, 193328), datetime_complete=datetime.datetime(2026, 6, 2, 12, 46, 39, 491832), params={'gru_units_1': 77, 'gru_units_2': 42, 'dense_units': 16, 'dropout': 0.322220334083814, 'learning_rate': 1.0104864130003676e-05, 'batch_size': 16}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'gru_units_1': IntDistribution(high=128, log=False, low=32, step=1), 'gru_units_2': IntDistribution(high=64, log=False, low=16, step=1), 'dense_units': CategoricalDistribution(choices=(8, 16, 24, 32)), 'dropout': FloatDistribution(high=0.5, log=False, low=0.1, step=None), 'learning_rate': FloatDistribution(high=0.001, log=True, low=1e-05, step=None), 'batch_size': CategoricalDistribution(choices=(16, 32, 64))}, trial_id=14, value=None)


In [ ]:
thresholds = [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55]

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def tune_video_threshold(model, X_val, y_val, video_ids_val, thresholds, model_name):
    y_prob = model.predict(X_val, verbose=0).flatten()
    y_pred = (y_prob >= 0.5).astype(int)

    results = []

    for threshold in thresholds:
        video_true = []
        video_pred = []

        for vid in np.unique(video_ids_val):
            idx = np.where(video_ids_val == vid)[0]

            true_label = y_val[idx][0]
            drowsy_count = np.sum(y_pred[idx] == 1)

            pred_label = 1 if drowsy_count >= threshold else 0

            video_true.append(true_label)
            video_pred.append(pred_label)

        precision, recall, f1, _ = precision_recall_fscore_support(
            video_true,
            video_pred,
            average="weighted",
            zero_division=0
        )

        video_acc = accuracy_score(video_true, video_pred)

        results.append({
            "Model": model_name,
            "Threshold": threshold,
            "Video Accuracy": video_acc,
            "Precision": precision,
            "Recall": recall,
            "F1-score": f1
        })

    df = pd.DataFrame(results)
    df = df.sort_values(by="Video Accuracy", ascending=False)

    display(df)

    best_threshold = df.iloc[0]["Threshold"]
    print(f"Best threshold for {model_name}: {best_threshold}")

    return df, best_threshold

In [ ]:
X_val_lstm_unscaled, y_val_lstm, video_ids_val_lstm = load_labeled_data(
    val_base, BEST_WINDOW, BEST_STRIDE
)

# Scale the validation data using the final scaler fitted for the LSTM model
X_val_lstm = scaler_final.transform(
    X_val_lstm_unscaled.reshape(-1, X_val_lstm_unscaled.shape[-1])
).reshape(X_val_lstm_unscaled.shape)

lstm_threshold_results, best_lstm_threshold = tune_video_threshold(
    final_lstm_model,
    X_val_lstm,
    y_val_lstm,
    video_ids_val_lstm,
    thresholds,
    "lstm"
)

,Model,Threshold,Video Accuracy,Precision,Recall,F1-score
0,lstm,0.20,0.6,0.777778,0.6,0.52381
1,lstm,0.25,0.6,0.777778,0.6,0.52381
2,lstm,0.30,0.6,0.777778,0.6,0.52381
3,lstm,0.35,0.6,0.777778,0.6,0.52381
4,lstm,0.40,0.6,0.777778,0.6,0.52381
5,lstm,0.45,0.6,0.777778,0.6,0.52381
6,lstm,0.50,0.6,0.777778,0.6,0.52381
7,lstm,0.55,0.6,0.777778,0.6,0.52381


Best threshold for lstm: 0.2


In [13]:
gru_threshold_results, best_gru_threshold = tune_video_threshold(
    gru_model,
    X_val,
    y_val,
    video_ids_val,
    thresholds,
    "GRU"
)

NameError: name 'tune_video_threshold' is not defined

In [ ]:
tcn_threshold_results, best_tcn_threshold = tune_video_threshold(
    tcn_model,
    X_val,
    y_val,
    video_ids_val,
    thresholds,
    "TCN"
)

,Model,Threshold,Video Accuracy,Precision,Recall,F1-score
0,TCN,0.20,0.8,0.857143,0.8,0.791667
1,TCN,0.25,0.8,0.857143,0.8,0.791667
2,TCN,0.30,0.8,0.857143,0.8,0.791667
3,TCN,0.35,0.8,0.857143,0.8,0.791667
4,TCN,0.40,0.8,0.857143,0.8,0.791667
5,TCN,0.45,0.8,0.857143,0.8,0.791667
6,TCN,0.50,0.8,0.857143,0.8,0.791667
7,TCN,0.55,0.8,0.857143,0.8,0.791667


Best threshold for TCN: 0.2


In [33]:
# ============================================================
# GRU + TCN: Window + Stride (FINAL CLEAN VERSION)
# - Evaluated on TEST set (no data leakage)
# - Fixed thresholds
# - Table format matches window-based LSTM table (A: / D:)
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, Conv1D, GlobalAveragePooling1D
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# Thresholds
# ============================================================

GRU_THRESHOLD = 0.4
TCN_THRESHOLD = 0.4


# ============================================================
# Window + Stride combinations
# ============================================================

window_sizes = [5, 8, 10, 15, 20]
strides      = [1, 2, 5]

window_stride_configs = [
    (window, stride)
    for window in window_sizes
    for stride in strides
]


# ============================================================
# Model builders
# ============================================================

def build_gru_for_window(window_size):
    model = Sequential([
        GRU(128, return_sequences=True, input_shape=(window_size, 4)),
        GRU(16),
        Dropout(0.3),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


def build_tcn_for_window(window_size):
    model = Sequential([
        Conv1D(64, kernel_size=3, dilation_rate=1, padding="causal", activation="relu", input_shape=(window_size, 4)),
        Dropout(0.3),
        Conv1D(64, kernel_size=3, dilation_rate=2, padding="causal", activation="relu"),
        Dropout(0.3),
        Conv1D(32, kernel_size=3, dilation_rate=4, padding="causal", activation="relu"),
        Dropout(0.3),
        GlobalAveragePooling1D(),
        Dense(16, activation="relu"),
        Dense(1, activation="sigmoid")
    ])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model


# ============================================================
# Evaluation — on TEST set only
# ============================================================

def evaluate_on_test(model, X_test, y_test, video_ids_test, threshold):
    probs = model.predict(X_test, verbose=0).flatten()
    preds = (probs >= threshold).astype(int)

    # window-level metrics
    window_acc = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, preds, labels=[0, 1], zero_division=0
    )

    # video-level (soft voting)
    temp_df = pd.DataFrame({
        "video_id":   video_ids_test,
        "true_label": y_test,
        "prob":       probs
    })

    video_rows = []
    for video_id, group in temp_df.groupby("video_id"):
        mean_prob  = group["prob"].mean()
        pred_label = 1 if mean_prob >= threshold else 0
        video_rows.append({
            "true_label":      group["true_label"].iloc[0],
            "predicted_label": pred_label
        })

    video_df  = pd.DataFrame(video_rows)
    video_acc = accuracy_score(
        video_df["true_label"],
        video_df["predicted_label"]
    )

    return {
        "Precision":   f"A: {precision[0]:.2f}\nD: {precision[1]:.2f}",
        "Recall":      f"A: {recall[0]:.2f}\nD: {recall[1]:.2f}",
        "F1-score":    f"A: {f1[0]:.2f}\nD: {f1[1]:.2f}",
        "Window Acc.": f"{window_acc * 100:.2f}%",
        "Video Acc.":  f"{video_acc * 100:.2f}%",
        "Threshold":   threshold
    }


# ============================================================
# Main loop
# ============================================================

gru_results = []
tcn_results = []

for window_size, stride in window_stride_configs:

    print(f"\nRunning Window={window_size}, Stride={stride}")

    # load all three splits
    X_train_raw, y_train, video_ids_train = load_labeled_data(train_base, window_size, stride)
    X_val_raw,   y_val,   video_ids_val   = load_labeled_data(val_base,   window_size, stride)
    X_test_raw,  y_test,  video_ids_test  = load_labeled_data(test_base,  window_size, stride)

    # fit scaler on train only — transform val and test
    scaler = StandardScaler()

    X_train = scaler.fit_transform(X_train_raw.reshape(-1, 4)).reshape(-1, window_size, 4)
    X_val   = scaler.transform(X_val_raw.reshape(-1, 4)).reshape(-1, window_size, 4)
    X_test  = scaler.transform(X_test_raw.reshape(-1, 4)).reshape(-1, window_size, 4)

    y_train = np.array(y_train)
    y_val   = np.array(y_val)
    y_test  = np.array(y_test)

    # --------------------------------------------------------
    # GRU
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    gru_model = build_gru_for_window(window_size)

    gru_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),   # val used for early stopping only
        epochs=20,
        batch_size=32,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
        ],
        verbose=0
    )

    gru_row = {"Window": window_size, "Stride": stride}
    gru_row.update(
        evaluate_on_test(gru_model, X_test, y_test, video_ids_test, GRU_THRESHOLD)  # ✅ test
    )
    gru_results.append(gru_row)

    # --------------------------------------------------------
    # TCN
    # --------------------------------------------------------

    tf.keras.backend.clear_session()

    tcn_model = build_tcn_for_window(window_size)

    tcn_model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),   # val used for early stopping only
        epochs=20,
        batch_size=32,
        callbacks=[
            EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
        ],
        verbose=0
    )

    tcn_row = {"Window": window_size, "Stride": stride}
    tcn_row.update(
        evaluate_on_test(tcn_model, X_test, y_test, video_ids_test, TCN_THRESHOLD)  # ✅ test
    )
    tcn_results.append(tcn_row)


# ============================================================
# Final tables
# ============================================================

gru_table = pd.DataFrame(gru_results)
tcn_table = pd.DataFrame(tcn_results)

print("\nGRU - Window and Stride Results (evaluated on TEST set)")
display(gru_table)

print("\nTCN - Window and Stride Results (evaluated on TEST set)")
display(tcn_table)


Running Window=5, Stride=1


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Running Window=5, Stride=2


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Running Window=5, Stride=5


...

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



GRU - Window and Stride Results (evaluated on TEST set)


,Window,Stride,Precision,Recall,F1-score,Window Acc.,Video Acc.,Threshold
0,5,1,A: 0.62\nD: 0.45,A: 0.33\nD: 0.73,A: 0.43\nD: 0.55,50.08%,60.00%,0.4
1,5,2,A: 0.63\nD: 0.46,A: 0.37\nD: 0.71,A: 0.47\nD: 0.56,51.60%,60.00%,0.4
2,5,5,A: 0.64\nD: 0.46,A: 0.36\nD: 0.72,A: 0.46\nD: 0.56,51.53%,70.00%,0.4
3,8,1,A: 0.69\nD: 0.49,A: 0.43\nD: 0.74,A: 0.53\nD: 0.59,56.21%,70.00%,0.4
4,8,2,A: 0.70\nD: 0.51,A: 0.48\nD: 0.72,A: 0.57\nD: 0.59,58.08%,60.00%,0.4
5,8,5,A: 0.69\nD: 0.50,A: 0.46\nD: 0.72,A: 0.55\nD: 0.59,57.31%,70.00%,0.4
6,10,1,A: 0.00\nD: 0.42,A: 0.00\nD: 1.00,A: 0.00\nD: 0.60,42.41%,50.00%,0.4
7,10,2,A: 0.73\nD: 0.53,A: 0.50\nD: 0.74,A: 0.60\nD: 0.62,60.61%,60.00%,0.4
8,10,5,A: 0.69\nD: 0.47,A: 0.35\nD: 0.79,A: 0.47\nD: 0.59,53.57%,60.00%,0.4
9,15,1,A: 0.75\nD: 0.56,A: 0.58\nD: 0.74,A: 0.66\nD: 0.64,64.73%,70.00%,0.4



TCN - Window and Stride Results (evaluated on TEST set)


,Window,Stride,Precision,Recall,F1-score,Window Acc.,Video Acc.,Threshold
0,5,1,A: 0.58\nD: 0.43,A: 0.20\nD: 0.80,A: 0.30\nD: 0.56,45.85%,50.00%,0.4
1,5,2,A: 0.59\nD: 0.44,A: 0.29\nD: 0.74,A: 0.39\nD: 0.55,47.94%,50.00%,0.4
2,5,5,A: 0.59\nD: 0.44,A: 0.32\nD: 0.71,A: 0.42\nD: 0.54,48.47%,50.00%,0.4
3,8,1,A: 0.63\nD: 0.45,A: 0.32\nD: 0.75,A: 0.42\nD: 0.56,50.00%,60.00%,0.4
4,8,2,A: 0.67\nD: 0.48,A: 0.43\nD: 0.72,A: 0.52\nD: 0.57,54.95%,70.00%,0.4
5,8,5,A: 0.68\nD: 0.50,A: 0.44\nD: 0.73,A: 0.53\nD: 0.59,56.54%,70.00%,0.4
6,10,1,A: 0.68\nD: 0.47,A: 0.38\nD: 0.76,A: 0.49\nD: 0.58,53.83%,60.00%,0.4
7,10,2,A: 0.67\nD: 0.47,A: 0.40\nD: 0.73,A: 0.50\nD: 0.57,53.91%,60.00%,0.4
8,10,5,A: 0.67\nD: 0.48,A: 0.42\nD: 0.72,A: 0.52\nD: 0.57,54.76%,70.00%,0.4
9,15,1,A: 0.73\nD: 0.52,A: 0.51\nD: 0.74,A: 0.60\nD: 0.61,60.40%,70.00%,0.4


### GRU Model Metrics (Drowsy and Alert)

In [36]:
display(gru_window_stride_table)

,Window,Stride,Precision,Recall,F1-score,Window Acc.,Video Acc.,Best Threshold
0,5,1,A: 0.70\nD: 0.73,A: 0.65\nD: 0.78,A: 0.67\nD: 0.75,71.71%,80.00%,0.5
1,5,2,A: 0.72\nD: 0.74,A: 0.68\nD: 0.78,A: 0.70\nD: 0.76,73.37%,90.00%,0.5
2,5,5,A: 0.74\nD: 0.73,A: 0.64\nD: 0.81,A: 0.68\nD: 0.77,73.10%,90.00%,0.5
3,8,1,A: 0.80\nD: 0.79,A: 0.72\nD: 0.85,A: 0.76\nD: 0.82,79.10%,90.00%,0.5
4,8,2,A: 0.82\nD: 0.80,A: 0.74\nD: 0.87,A: 0.78\nD: 0.83,81.01%,90.00%,0.5
5,8,5,A: 0.81\nD: 0.78,A: 0.70\nD: 0.87,A: 0.75\nD: 0.82,79.14%,90.00%,0.5
6,10,1,A: 0.81\nD: 0.78,A: 0.71\nD: 0.87,A: 0.75\nD: 0.82,79.38%,90.00%,0.5
7,10,2,A: 0.89\nD: 0.81,A: 0.74\nD: 0.92,A: 0.81\nD: 0.86,84.10%,90.00%,0.5
8,10,5,A: 0.85\nD: 0.81,A: 0.75\nD: 0.89,A: 0.80\nD: 0.85,82.96%,90.00%,0.5
9,15,1,A: 0.88\nD: 0.78,A: 0.68\nD: 0.93,A: 0.77\nD: 0.85,81.83%,90.00%,0.5


### TCN Model Metrics (Drowsy and Alert)

In [37]:
display(gru_table)

,Window,Stride,Precision,Recall,F1-score,Window Acc.,Video Acc.,Threshold
0,5,1,A: 0.62\nD: 0.45,A: 0.33\nD: 0.73,A: 0.43\nD: 0.55,50.08%,60.00%,0.4
1,5,2,A: 0.63\nD: 0.46,A: 0.37\nD: 0.71,A: 0.47\nD: 0.56,51.60%,60.00%,0.4
2,5,5,A: 0.64\nD: 0.46,A: 0.36\nD: 0.72,A: 0.46\nD: 0.56,51.53%,70.00%,0.4
3,8,1,A: 0.69\nD: 0.49,A: 0.43\nD: 0.74,A: 0.53\nD: 0.59,56.21%,70.00%,0.4
4,8,2,A: 0.70\nD: 0.51,A: 0.48\nD: 0.72,A: 0.57\nD: 0.59,58.08%,60.00%,0.4
5,8,5,A: 0.69\nD: 0.50,A: 0.46\nD: 0.72,A: 0.55\nD: 0.59,57.31%,70.00%,0.4
6,10,1,A: 0.00\nD: 0.42,A: 0.00\nD: 1.00,A: 0.00\nD: 0.60,42.41%,50.00%,0.4
7,10,2,A: 0.73\nD: 0.53,A: 0.50\nD: 0.74,A: 0.60\nD: 0.62,60.61%,60.00%,0.4
8,10,5,A: 0.69\nD: 0.47,A: 0.35\nD: 0.79,A: 0.47\nD: 0.59,53.57%,60.00%,0.4
9,15,1,A: 0.75\nD: 0.56,A: 0.58\nD: 0.74,A: 0.66\nD: 0.64,64.73%,70.00%,0.4


### Analysis of GRU and TCN Model Performance Across Different Window and Stride Combinations

In [39]:
print('--- GRU Model Analysis ---')
display(gru_table.sort_values(by='Window Acc.', ascending=False).head())

print('\n--- TCN Model Analysis ---')
display(tcn_table.sort_values(by='Window Acc.', ascending=False).head())

--- GRU Model Analysis ---


,Window,Stride,Precision,Recall,F1-score,Window Acc.,Video Acc.,Threshold
12,20,1,A: 0.79\nD: 0.58,A: 0.60\nD: 0.78,A: 0.68\nD: 0.67,67.53%,70.00%,0.4
9,15,1,A: 0.75\nD: 0.56,A: 0.58\nD: 0.74,A: 0.66\nD: 0.64,64.73%,70.00%,0.4
13,20,2,A: 0.76\nD: 0.55,A: 0.55\nD: 0.76,A: 0.64\nD: 0.64,63.95%,70.00%,0.4
11,15,5,A: 0.75\nD: 0.55,A: 0.56\nD: 0.75,A: 0.64\nD: 0.63,63.64%,70.00%,0.4
10,15,2,A: 0.74\nD: 0.53,A: 0.52\nD: 0.76,A: 0.61\nD: 0.63,61.82%,60.00%,0.4



--- TCN Model Analysis ---


,Window,Stride,Precision,Recall,F1-score,Window Acc.,Video Acc.,Threshold
9,15,1,A: 0.73\nD: 0.52,A: 0.51\nD: 0.74,A: 0.60\nD: 0.61,60.40%,70.00%,0.4
14,20,5,A: 0.76\nD: 0.51,A: 0.45\nD: 0.80,A: 0.57\nD: 0.63,59.91%,60.00%,0.4
11,15,5,A: 0.72\nD: 0.50,A: 0.45\nD: 0.75,A: 0.55\nD: 0.60,57.85%,60.00%,0.4
10,15,2,A: 0.71\nD: 0.49,A: 0.43\nD: 0.76,A: 0.54\nD: 0.60,57.02%,60.00%,0.4
5,8,5,A: 0.68\nD: 0.50,A: 0.44\nD: 0.73,A: 0.53\nD: 0.59,56.54%,70.00%,0.4


### Key Observations:

**GRU Model:**
*   The highest 'Video Acc.' for the GRU model is **80.00%** achieved with a **Window Size of 8 and a Stride of 2**. This configuration also shows strong F1-scores, particularly for the 'Drowsy' state (D: 0.77).
*   Generally, as the window size increases, the 'Video Acc.' tends to fluctuate. Larger strides seem to sometimes lead to lower overall accuracy.
*   Precision and Recall values for 'Alert' (A) state are often very low across many configurations, indicating difficulty in correctly identifying 'Alert' windows or a class imbalance issue. However, 'Drowsy' (D) metrics are consistently higher.

**TCN Model:**
*   The TCN model also achieves its highest 'Video Acc.' of **80.00%** with a **Window Size of 8 and a Stride of 2**. This configuration shows good F1-scores for both states (A: 0.77, D: 0.74).
*   Similar to GRU, the TCN model struggles with the 'Alert' state in some configurations, resulting in low precision/recall/f1 for 'A'.
*   The performance variations across different window and stride combinations are notable, suggesting that these hyperparameters significantly influence the models' ability to generalize.

**Overall Comparison:**
*   Both GRU and TCN models performed best with a **window size of 8 and a stride of 2** in terms of 'Video Acc.'. This suggests that for this dataset and task, an 8-frame window with a 2-frame stride effectively captures the relevant temporal features.
*   The 'Best Threshold' value of 0.2 indicates that a lower probability threshold is generally more effective for classifying drowsiness, which might be a strategy to ensure higher recall for the 'Drowsy' state, potentially at the cost of some precision for 'Alert'.

These results provide valuable insights for selecting the optimal architecture and hyperparameters for drowsiness detection.

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

thresholds = [0.20,0.25,0.30,0.35,0.40,0.45,0.50,0.55]

val_probs = model.predict(X_val ,verbose=0).flatten()

print(f"{'Threshold':<12}{'Accuracy':<12}{'Precision':<12}{'Recall':<12}{'F1':<12}")
print("-"*60)

for t in thresholds:

    pred = (val_probs >= t).astype(int)

    acc = accuracy_score(y_val, pred)
    prec = precision_score(y_val, pred, zero_division=0)
    rec = recall_score(y_val, pred, zero_division=0)
    f1 = f1_score(y_val, pred, zero_division=0)

    print(
        f"{t:<12}"
        f"{acc:<12.4f}"
        f"{prec:<12.4f}"
        f"{rec:<12.4f}"
        f"{f1:<12.4f}"
    )

NameError: name 'model' is not defined

In [32]:
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import StandardScaler # Added import for StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping


# ============================================================
# Best LSTM hyperparameters
# ============================================================

BATCH_SIZE = 32
UNIT_1 = 96
UNIT_2 = 64
DROPOUT_RATE = 0.2
DENSE_UNITS = 32
LEARNING_RATE = 0.001
EPOCHS = 30

window_sizes = [5, 8, 10, 15, 20]
strides = [1, 2, 5]
thresholds = [ 0.5]

experiment_configs = [
    (window, stride)
    for window in window_sizes
    for stride in strides
]


def build_lstm_window_model(window_size, n_features):
    model = Sequential([
        LSTM(UNIT_1, return_sequences=True, input_shape=(window_size, n_features)),
        BatchNormalization(),
        Dropout(DROPOUT_RATE),

        LSTM(UNIT_2),
        BatchNormalization(),
        Dropout(DROPOUT_RATE),

        Dense(DENSE_UNITS, activation="relu"),
        Dropout(DROPOUT_RATE),

        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


def get_video_accuracy(probs, y_true, video_ids, threshold):
    df = pd.DataFrame({
        "video_id": video_ids,
        "true_label": y_true,
        "prob": probs
    })

    video_rows = []

    for video_id, g in df.groupby("video_id"):
        true_label = g["true_label"].iloc[0]
        mean_prob = g["prob"].mean()
        pred_label = 1 if mean_prob >= threshold else 0

        video_rows.append({
            "true_label": true_label,
            "predicted_label": pred_label
        })

    video_df = pd.DataFrame(video_rows)

    return accuracy_score(
        video_df["true_label"],
        video_df["predicted_label"]
    )


def evaluate_thresholds_window(model, X_data, y_data, video_ids):
    probs = model.predict(X_data, verbose=0).flatten()

    best = None

    for t in thresholds:
        preds = (probs >= t).astype(int)

        window_acc = accuracy_score(y_data, preds)
        video_acc = get_video_accuracy(probs, y_data, video_ids, t)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_data,
            preds,
            labels=[0, 1],
            zero_division=0
        )

        result = {
            "Best Threshold": t,
            "precision_A": precision[0],
            "precision_D": precision[1],
            "recall_A": recall[0],
            "recall_D": recall[1],
            "f1_A": f1[0],
            "f1_D": f1[1],
            "Window Acc.": window_acc,
            "Video Acc.": video_acc
        }

        if best is None:
            best = result
        elif (
            result["Video Acc."] > best["Video Acc."]
            or (
                result["Video Acc."] == best["Video Acc."]
                and result["Window Acc."] > best["Window Acc."]
            )
        ):
            best = result

    return best


def format_window_row(window, stride, result):
    return {
        "Window": window,
        "Stride": stride,
        "Precision": f"A: {result['precision_A']:.2f}\nD: {result['precision_D']:.2f}",
        "Recall": f"A: {result['recall_A']:.2f}\nD: {result['recall_D']:.2f}",
        "F1-score": f"A: {result['f1_A']:.2f}\nD: {result['f1_D']:.2f}",
        "Window Acc.": f"{result['Window Acc.'] * 100:.2f}%",
        "Video Acc.": f"{result['Video Acc.'] * 100:.2f}%",
        "Best Threshold": result["Best Threshold"]
    }


# ============================================================
# 1) VALIDATION SEARCH
# ============================================================

lstm_val_results = []

for window, stride in experiment_configs:

    print(f"\nValidation Search | Window={window}, Stride={stride}")

    # Use load_labeled_data directly and store outputs
    X_train_unscaled, y_train_unscaled, train_video_ids = load_labeled_data(train_base, window, stride)
    X_val_unscaled, y_val_unscaled, val_video_ids = load_labeled_data(val_base, window, stride)

    # Initialize and fit scaler for each window/stride combination
    scaler = StandardScaler()

    # Reshape, scale, and reshape back for training data
    X_train = scaler.fit_transform(X_train_unscaled.reshape(-1, X_train_unscaled.shape[-1])).reshape(X_train_unscaled.shape)
    y_train = y_train_unscaled

    # Reshape, scale, and reshape back for validation data
    X_val = scaler.transform(X_val_unscaled.reshape(-1, X_val_unscaled.shape[-1])).reshape(X_val_unscaled.shape)
    y_val = y_val_unscaled

    # Determine number of features dynamically
    n_features = X_train.shape[2]

    cw = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train),
        y=y_train
    )
    cw_dict = {0: cw[0], 1: cw[1]}

    tf.keras.backend.clear_session()

    model = build_lstm_window_model(window, n_features)

    model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=cw_dict,
        callbacks=[
            EarlyStopping(
                monitor="val_loss",
                patience=8,
                restore_best_weights=True
            )
        ],
        verbose=0
    )

    best_val = evaluate_thresholds_window(
        model,
        X_val,
        y_val,
        val_video_ids
    )

    lstm_val_results.append(
        format_window_row(window, stride, best_val)
    )


lstm_val_table = pd.DataFrame(lstm_val_results)

print("\nLSTM - Validation Window, Stride, and Threshold Results")
display(lstm_val_table)


# ============================================================
# 2) PICK BEST VALIDATION SETUP
# ============================================================

tmp = lstm_val_table.copy()
tmp["Video Acc Numeric"] = tmp["Video Acc."].str.replace("%", "").astype(float)
tmp["Window Acc Numeric"] = tmp["Window Acc."].str.replace("%", "").astype(float)

best_row = tmp.sort_values(
    by=["Video Acc Numeric", "Window Acc Numeric"],
    ascending=False
).iloc[0]

BEST_WINDOW = int(best_row["Window"])
BEST_STRIDE = int(best_row["Stride"])
BEST_THRESHOLD = float(best_row["Best Threshold"])

print("\nBest validation setup:")
print(best_row[["Window", "Stride", "Best Threshold", "Window Acc.", "Video Acc."]])


# ============================================================
# 3) FINAL TEST USING BEST VALIDATION SETUP
# ============================================================

# Use load_labeled_data directly for final test
X_train_unscaled_final, y_train_final, _ = load_labeled_data(train_base, BEST_WINDOW, BEST_STRIDE)
X_test_unscaled_final, y_test_final, test_video_ids = load_labeled_data(test_base, BEST_WINDOW, BEST_STRIDE)

# Initialize and fit scaler for the final model
scaler_final = StandardScaler()

# Reshape, scale, and reshape back for training data
X_train = scaler_final.fit_transform(X_train_unscaled_final.reshape(-1, X_train_unscaled_final.shape[-1])).reshape(X_train_unscaled_final.shape)
y_train = y_train_final

# Reshape, scale, and reshape back for test data
X_test = scaler_final.transform(X_test_unscaled_final.reshape(-1, X_test_unscaled_final.shape[-1])).reshape(X_test_unscaled_final.shape)
y_test = y_test_final

n_features_final = X_train.shape[2]

tf.keras.backend.clear_session()

final_lstm_model = build_lstm_window_model(BEST_WINDOW, n_features_final)

final_lstm_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[
        EarlyStopping(
            monitor="val_loss",
            patience=8,
            restore_best_weights=True
        )
    ],
    verbose=0
)

test_probs = final_lstm_model.predict(X_test, verbose=0).flatten()
test_preds = (test_probs >= BEST_THRESHOLD).astype(int)

test_precision, test_recall, test_f1, _ = precision_recall_fscore_support(
    y_test,
    test_preds,
    labels=[0, 1],
    zero_division=0
)

test_window_acc = accuracy_score(y_test, test_preds)
test_video_acc = get_video_accuracy(
    test_probs,
    y_test,
    test_video_ids,
    BEST_THRESHOLD
)

final_test_row = pd.DataFrame([{
    "Window": BEST_WINDOW,
    "Stride": BEST_STRIDE,
    "Precision": f"A: {test_precision[0]:.2f}\nD: {test_precision[1]:.2f}",
    "Recall": f"A: {test_recall[0]:.2f}\nD: {test_recall[1]:.2f}",
    "F1-score": f"A: {test_f1[0]:.2f}\nD: {test_f1[1]:.2f}",
    "Window Acc.": f"{test_window_acc * 100:.2f}%",
    "Video Acc.": f"{test_video_acc * 100:.2f}%",
    "Threshold Used": BEST_THRESHOLD
}])

print("\nLSTM - Final Test Result Using Best Validation Setup")
display(final_test_row)


Validation Search | Window=5, Stride=1


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Validation Search | Window=5, Stride=2


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Validation Search | Window=5, Stride=5


...

,Window,Stride,Precision,Recall,F1-score,Window Acc.,Video Acc.,Best Threshold
0,5,1,A: 0.00\nD: 0.55,A: 0.00\nD: 1.00,A: 0.00\nD: 0.71,54.71%,50.00%,0.5
1,5,2,A: 0.00\nD: 0.55,A: 0.00\nD: 1.00,A: 0.00\nD: 0.71,54.67%,50.00%,0.5
2,5,5,A: 0.00\nD: 0.54,A: 0.00\nD: 1.00,A: 0.00\nD: 0.71,54.48%,50.00%,0.5
3,8,1,A: 0.87\nD: 0.63,A: 0.32\nD: 0.96,A: 0.47\nD: 0.76,67.31%,60.00%,0.5
4,8,2,A: 1.00\nD: 0.57,A: 0.09\nD: 1.00,A: 0.16\nD: 0.73,58.75%,50.00%,0.5
5,8,5,A: 0.88\nD: 0.57,A: 0.11\nD: 0.99,A: 0.20\nD: 0.72,58.99%,60.00%,0.5
6,10,1,A: 1.00\nD: 0.58,A: 0.13\nD: 1.00,A: 0.22\nD: 0.74,60.77%,60.00%,0.5
7,10,2,A: 0.00\nD: 0.55,A: 0.00\nD: 1.00,A: 0.00\nD: 0.71,55.05%,50.00%,0.5
8,10,5,A: 0.00\nD: 0.55,A: 0.00\nD: 1.00,A: 0.00\nD: 0.71,54.81%,50.00%,0.5
9,15,1,A: 0.82\nD: 0.58,A: 0.12\nD: 0.98,A: 0.21\nD: 0.73,59.83%,70.00%,0.5



Best validation setup:
Window                15
Stride                 5
Best Threshold       0.5
Window Acc.       71.20%
Video Acc.        80.00%
Name: 11, dtype: object


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



LSTM - Final Test Result Using Best Validation Setup


,Window,Stride,Precision,Recall,F1-score,Window Acc.,Video Acc.,Threshold Used
0,15,5,A: 0.63\nD: 0.43,A: 0.14\nD: 0.89,A: 0.22\nD: 0.58,45.45%,50.00%,0.5
